# INFO3020 Week 3: Data Quality and Missing Values

Dataset: published 2023 Vietnamese high school graduation exam scores. The assignment is on slide 35 of *W3 - Data Quality and Processing_.pptx*. This notebook regenerates the full audit, then reads the tables and figures. See `../docs/report+bao_cao.md` for the written EX3.1–EX3.3 arguments. The CSV is read only.


## 1. Run from the raw file

The source ID stays a string, preserving leading zeros. The pipeline writes a Parquet copy with published scores and nulls, plus audit tables and experiment outputs. It does not write imputed scores to Parquet.


In [1]:
from pathlib import Path
import json
import sys
import pandas as pd
root = Path.cwd().resolve()
if not (root / 'src').exists:
    root = root.parent
sys.path.insert(0, str(root))
from src.analyze import run, ROOT

results = run()
print('CSV rows:', results['audit']['rows'])
print('Columns:', results['audit']['columns'])
print('Raw SHA-256:', results['provenance']['source_sha256'])
print('Raw token types:', results['provenance']['raw_tokens'])


CSV rows: 1022060
Columns: 11
Raw SHA-256: cdf22a6b45f8e23b522beb1c521782e36486cac39d3fb64bca2a2395edec39b5
Raw token types: {'Biology | blank_or_whitespace': 697435, 'Chemistry | blank_or_whitespace': 693942, 'Civic education | blank_or_whitespace': 456608, 'Foreign language | blank_or_whitespace': 141063, 'Foreign language code | blank_or_whitespace': 141063, 'Geography | blank_or_whitespace': 339926, 'History | blank_or_whitespace': 338613, 'Literature | blank_or_whitespace': 13821, 'Mathematics | blank_or_whitespace': 18687, 'Physics | blank_or_whitespace': 694871}


## 2. EX3.1: audit six dimensions

Slides 7 and 11 define the dimensions and suggest a reusable column profile. Scores are reasoned judgments for describing *published 2023 scores*, not measured accuracy percentages. A value within 0–10 passes a validity check; it cannot verify the original exam result.


In [2]:
profile = pd.read_csv(ROOT / 'outputs/tables/column_profile+ho_so_cot.csv')
print(profile[['column', 'dtype', 'n_observed', 'n_missing', 'pct_missing', 'n_unique_observed', 'min', 'max']].to_string(index=False, float_format=lambda x: f'{x:.3f}'))


               column   dtype  n_observed  n_missing  pct_missing  n_unique_observed   min    max
           Student ID  string     1022060          0        0.000            1022060   NaN    NaN
          Mathematics float64     1003373      18687        1.828                 51 0.000 10.000
           Literature float64     1008239      13821        1.352                215 0.000 10.000
     Foreign language float64      880997     141063       13.802                 50 0.000 10.000
              Physics float64      327189     694871       67.987                 41 0.000 10.000
            Chemistry float64      328118     693942       67.896                 39 0.000 10.000
              Biology float64      324625     697435       68.238                 41 0.000 10.000
              History float64      683447     338613       33.130                 41 0.000 10.000
            Geography float64      682134     339926       33.259                 40 0.000 10.000
      Civic educatio

In [3]:
quality = pd.DataFrame(results['quality_scores'])
print(quality.to_string(index=False, max_colwidth=72))
print('Duplicate ID rows:', results['audit']['student_id_duplicate_extra_rows'])
print('Score cells outside 0–10:', sum(results['audit']['out_of_range_or_nonfinite_scores'].values()))
print('Language score/code state mismatches:', results['audit']['language_score_code_mismatch'])


   dimension  score_1_to_5                                                                 evidence                                                               limitation
Completeness             3 Math missing 18687/1022060; 683813 have social scores only; 8817 have... Registration and exemption records are unavailable; true required-sco...
    Accuracy             2 No independent official result table or candidate-level reconciliatio... In-range scores alone cannot establish that a published score matches...
 Consistency             4 Foreign-language score/code state mismatches 0; both composite groups...              Unusual partial subject patterns need registration metadata
    Validity             5 Out-of-range/nonfinite score cells 0; invalid language codes 0; inval... Language-code set follows dataset documentation and project mapping, ...
  Uniqueness             5               Extra repeated Student IDs 0; extra whole-row duplicates 0                 Student ID is treat

![Missing percentage across all candidates](../outputs/figures/missing_rates+ty_le_thieu.png)

The bar chart uses the full CSV. High missingness in an elective subject may reflect the exam structure. It is not a data-error rate by itself.


## 3. EX3.2: source tokens, patterns and treatment

Slides 13–15 call for inspecting hidden missing tokens and co-occurring gaps. The group labels below mean *at least one published score in that group*. They do not prove exam registration or attendance.


In [4]:
tokens = pd.DataFrame([{'column_and_token': k, 'n': v} for k, v in results['provenance']['raw_tokens'].items()])
print(tokens.to_string(index=False))
observed_groups = pd.DataFrame([{'observed_exam_group': k, 'n': v, 'pct_all_rows': 100*v/results['audit']['rows']} for k, v in results['audit']['exam_groups'].items()])
print()
print('Observed subject groups:')
print(observed_groups.to_string(index=False, float_format=lambda x: f'{x:.3f}'))
print()
print('Cross-tab of published subject counts:')
print(pd.DataFrame(results['missing_patterns']['observed_subject_count_table']).to_string(index=False))


                           column_and_token      n
              Biology | blank_or_whitespace 697435
            Chemistry | blank_or_whitespace 693942
      Civic education | blank_or_whitespace 456608
     Foreign language | blank_or_whitespace 141063
Foreign language code | blank_or_whitespace 141063
            Geography | blank_or_whitespace 339926
              History | blank_or_whitespace 338613
           Literature | blank_or_whitespace  13821
          Mathematics | blank_or_whitespace  18687
              Physics | blank_or_whitespace 694871

Observed subject groups:
observed_exam_group      n  pct_all_rows
       science_only 329430        32.232
        social_only 683813        66.905
      both_observed      0         0.000
   neither_observed   8817         0.863

Cross-tab of published subject counts:
 science_observed_count      0    1      2      3
                      0   8817 1836 116734 565243
                      1   1468    0      0      0
                  

In [5]:
core = pd.DataFrame(results['missing_patterns']['core_missing_by_group'])
print('Missing Math, Literature and Foreign language by observed group:')
print(core.loc[core['column'].isin(['Mathematics', 'Literature', 'Foreign language'])].to_string(index=False, float_format=lambda x: f'{x:.3f}'))
print()
print('Elective-subject context:')
context = pd.DataFrame(results['missing_patterns']['missing_by_context'])
print(context[['column', 'observed_exam_group', 'group_n', 'missing_n']].to_string(index=False))


Missing Math, Literature and Foreign language by observed group:
          column observed_exam_group  group_n  missing_n  pct_within_group
     Mathematics        science_only   329430         33             0.010
     Mathematics         social_only   683813      13767             2.013
     Mathematics       both_observed        0          0               NaN
     Mathematics    neither_observed     8817       4887            55.427
      Literature        science_only   329430       8838             2.683
      Literature         social_only   683813        225             0.033
      Literature       both_observed        0          0               NaN
      Literature    neither_observed     8817       4758            53.964
Foreign language        science_only   329430      13916             4.224
Foreign language         social_only   683813     122024            17.845
Foreign language       both_observed        0          0               NaN
Foreign language    neither_observe

### Two denominators and two classes of blank scores

The raw view counts blanks among nine score columns for every candidate. The context view counts blanks among Mathematics, Literature, Foreign language and the three subjects of the **observed** composite group. It excludes candidates with no observed composite group. Opposite-group blanks are *structural candidates*, never confirmed non-applicable scores; all remaining blanks are unresolved. Only if a score should have existed does an MCAR/MAR/MNAR hypothesis apply.


In [6]:
layers = results['missing_layers']
raw = pd.DataFrame(layers['raw_nine_score_missing_counts'])
six = pd.DataFrame(layers['context_six_score_missing_counts'])
print('Raw missing scores among nine subjects:')
print(raw.groupby('missing_count')['n'].sum().to_string())
print('Context missing scores among six relevant subjects:')
print(six.pivot(index='missing_count', columns='observed_exam_group', values='n').to_string())
print('No observed composite group:', layers['neither_group_rows'])
print('Only Student ID remains:', layers['only_student_id_rows'])


Raw missing scores among nine subjects:
missing_count
0         0
1         0
2         0
3    872976
4     13763
5    105884
6     23898
7       610
8       453
9      4476
Context missing scores among six relevant subjects:
observed_exam_group  science_only  social_only
missing_count                                 
0                          313095       559881
1                            7850         5913
2                            2241       103643
3                            6168        14285
4                              71           87
5                               5            4
6                               0            0
No observed composite group: 8817
Only Student ID remains: 4476


In [7]:
classified = pd.DataFrame(layers['missing_context_classification'])
print(classified.to_string(index=False))
patterns = pd.read_csv(ROOT / 'outputs/tables/context_missing_subject_patterns+mau_mon_thieu.csv')
print('Most common exact missing-subject patterns (1-4 blanks, by group):')
print(patterns.loc[patterns.missing_count.between(1, 4)].groupby(['observed_exam_group', 'missing_count'], sort=False).head(3).to_string(index=False))
print('Social Civic/Foreign language context:', layers['civic_language_context'])


               column  missing_n  structural_candidate_opposite_group_n  unresolved_observed_group_n  unresolved_no_group_n  unresolved_total_n
          Mathematics      18687                                      0                        13800                   4887               18687
           Literature      13821                                      0                         9063                   4758               13821
     Foreign language     141063                                      0                       135940                   5123              141063
              Physics     694871                                 683813                         2241                   8817               11058
            Chemistry     693942                                 683813                         1312                   8817               10129
              Biology     697435                                 683813                         4805                   8817             

![Missingness matrix: 2,000 randomly sampled rows sorted by observed subject group](../outputs/figures/missing_matrix_sample+ma_tran_mau_o_thieu.png)

The matrix is only a picture of a seeded sample. All counts and percentages in the tables use the full CSV. The GDTX social exam in 2023 did not include Civic education, so a blank Civic score within the observed social group can also be structurally non-applicable; the CSV cannot confirm a candidate's program. MCAR/MAR/MNAR are hypotheses about a score that should have existed (slides 16–18). A subject that was never taken should not be imputed as a lost score.


In [8]:
treatment = pd.read_csv(ROOT / 'outputs/tables/missing_treatment+cach_xu_ly_thieu.csv')
print(treatment[['column', 'missing_n', 'missing_pct', 'mechanism_hypothesis', 'treatment', 'reason']].to_string(index=False, max_colwidth=62, float_format=lambda x: f'{x:.3f}'))


               column  missing_n  missing_pct                                           mechanism_hypothesis                                                      treatment                                                         reason
          Mathematics      18687        1.828 Unresolved; group association makes MCAR across all rows im...               Keep null; describe published Math scores with n Registration, absence and source-record status are unavaila...
           Literature      13821        1.352 Unresolved; group association makes MCAR across all rows im...         Keep null; describe published Literature scores with n Registration, absence and source-record status are unavaila...
     Foreign language     141063       13.802 Unresolved mixture of non-registration, exemption, absence ...          Keep null; report score and code missingness together Score and code are jointly blank; program, registration and...
              Physics     694871       67.987 Structural can

## 4. EX3.3: two ways to fill Mathematics

Slides 32 and 35 require the same column, n, mean, sample standard deviation and a distribution plot for two methods. The first comparison hypothetically fills the **18,687 naturally blank Math cells**. It does not assert that a Math score existed for every one of those candidates. The canonical Parquet retains nulls.


In [9]:
natural = pd.read_csv(ROOT / 'outputs/tables/natural_missing_sensitivity+do_nhay_thieu_tu_nhien.csv')
print(natural.to_string(index=False, float_format=lambda x: f'{x:.4f}'))
print('Group medians:', results['natural_missing_sensitivity']['group_medians'])
print('Fallback count:', results['natural_missing_sensitivity']['fallback_count'])


                    status  n_total  n_observed  n_missing   mean  std_sample  n_imputed  delta_mean_vs_published  delta_std_vs_published
       published_math_only  1022060     1003373      18687 6.2506      1.6333        NaN                      NaN                     NaN
hypothetical_global_median  1022060     1022060          0 6.2569      1.6190 18687.0000                   0.0064                 -0.0143
 hypothetical_group_median  1022060     1022060          0 6.2462      1.6194 18687.0000                  -0.0044                 -0.0140
Group medians: {'science_only': 7.6, 'social_only': 5.8}
Fallback count: 4887


![Math distribution under the hypothetical natural-missing sensitivity scenario](../outputs/figures/natural_missing_sensitivity+do_nhay_thieu_tu_nhien.png)

The second comparison hides 20% of **known** Math scores from a 50,000-row sample, then checks both methods against the held-out truth. Both use the same mask and donors. It measures performance for an artificial random gap, not for natural gaps whose causes are unknown.


In [10]:
experiment = pd.read_csv(ROOT / 'outputs/tables/imputation_comparison+so_sanh_dien_khuyet.csv')
print(experiment.to_string(index=False, float_format=lambda x: f'{x:.4f}'))
print('Fit medians:', results['experiment']['global_median'], results['experiment']['group_medians'])
print('Masked scores:', results['experiment']['artificially_masked'])
print('Fallback count:', results['experiment']['fallback_count'])


                  status  n_total  n_observed  n_missing   mean  std_sample  n_imputed  mae_masked  rmse_masked  delta_mean_vs_original  delta_std_vs_original
original_observed_sample    50000       50000          0 6.2564      1.6276        NaN         NaN          NaN                     NaN                    NaN
           after_masking    50000       40000      10000 6.2562      1.6283        NaN         NaN          NaN                     NaN                    NaN
           global_median    50000       50000          0 6.3250      1.4629 10000.0000      1.3166       1.6605                  0.0686                -0.1647
            group_median    50000       50000          0 6.2843      1.5057 10000.0000      1.1006       1.3958                  0.0279                -0.1219
Fit medians: 6.6 {'science_only': 7.6, 'social_only': 5.8}
Masked scores: 10000
Fallback count: 42


![Distribution after randomly masking 10,000 known Math scores](../outputs/figures/imputation_distribution+phan_phoi_dien_khuyet.png)

Both median methods change the score distribution. Compare the mean and standard deviation with the original sample, and the MAE/RMSE only on the masked cells. Group Median has lower error in this particular experiment, but that cannot establish the correct values of naturally missing scores.


## 5. Reconciliation and conclusion

The project's chosen treatment is to keep unverified missing exam scores as null. One-subject statistics use the observed scores and disclose n; two-subject statistics use pairs with both scores; a composite total requires every included score. Further metadata on subject registration, school program, exemptions and absences would be needed to classify candidate-level causes. Independent official scores would be needed to verify accuracy.


In [11]:
import hashlib
raw_sha = hashlib.sha256((ROOT / 'data/raw/original.csv').read_bytes()).hexdigest()
assert raw_sha == results['provenance']['source_sha256']
assert sum(results['audit']['exam_groups'].values()) == results['audit']['rows']
assert len(treatment) == sum(profile['n_missing'] > 0)
assert results['experiment']['artificially_masked'] == results['experiment']['fallback_count'] + sum(results['experiment']['group_fill_counts'].values())
assert results['natural_missing_sensitivity']['missing_n'] == results['audit']['missing']['Mathematics']['count']
assert sum(row['n'] for row in layers['raw_nine_score_missing_counts']) == results['audit']['rows']
assert sum(row['n'] for row in layers['context_six_score_missing_counts']) == results['audit']['exam_groups']['science_only'] + results['audit']['exam_groups']['social_only']
assert layers['only_student_id_rows'] == 4476
processed = pd.read_parquet(ROOT / 'data/processed/exam_2023.parquet')
assert list(processed.columns) == list(profile['column'])
assert len(processed) == results['audit']['rows']
assert processed['Mathematics'].isna().sum() == results['audit']['missing']['Mathematics']['count']
assert processed['Student ID'].iloc[0] == '01000001'
print('Reconciliation passed; published scores and natural nulls remain in Parquet.')


Reconciliation passed; published scores and natural nulls remain in Parquet.
